In [ ]:
"""
Week 3 - Day 3
Hyperparameter Tuning
======================
Finding optimal PPO hyperparameters
and comparing best PPO vs best DQN.

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from environment.pricing_env import DynamicPricingEnv
from training.ppo_hypertuner import (
    run_ppo_grid_search,
    plot_tuning_results
)
from training.combined_tuner import (
    find_best_combined
)
from config import PPO

plt.style.use('seaborn-v0_8')
print("✅ Tuning modules loaded!")
print(f"\nTesting {8} PPO configurations")

In [ ]:
env = DynamicPricingEnv()

print("Running PPO grid search...")
print("8 configs × 500 episodes each")
print("Takes about 5-8 minutes...\n")

best_config, results_df, best_agent = (
    run_ppo_grid_search(
        env,
        n_train=500,
        n_eval=50
    )
)

print(f"\n✅ Grid search complete!")
print(f"   Best: {best_config['label']}")

In [ ]:
plot_tuning_results(
    results_df,
    save_path='../results/ppo_tuning.png'
)

In [ ]:
print("=== PPO TUNING RESULTS ===\n")
print(results_df[[
    'Config', 'LR', 'Clip Range',
    'N Epochs', 'Mean Revenue'
]].to_string(index=False))

print(f"\n🏆 Best Config: {best_config['label']}")
print(f"   LR     : {best_config['learning_rate']}")
print(f"   Clip   : {best_config['clip_range']}")
print(f"   Epochs : {best_config['n_epochs']}")

In [ ]:
print("=== HYPERPARAMETER EXPLANATION ===\n")

params = {
    'Learning Rate (lr)': [
        'How fast network learns',
        'Too high → unstable',
        'Too low → slow learning',
        f"Best: {best_config['learning_rate']}"
    ],
    'Clip Range (ε)': [
        'Prevents too-large updates',
        'Too wide → unstable',
        'Too narrow → slow',
        f"Best: {best_config['clip_range']}"
    ],
    'N Epochs': [
        'How many times to reuse data',
        'More → better data efficiency',
        'Too many → overfitting',
        f"Best: {best_config['n_epochs']}"
    ],
    'Entropy Coef': [
        'Encourages exploration',
        'Higher → more random',
        'Lower → more greedy',
        f"Best: {best_config['ent_coef']}"
    ],
}

for param, info in params.items():
    print(f"  {param}:")
    for line in info:
        print(f"    → {line}")
    print()

In [ ]:
print("Finding best PPO vs best DQN...")
print("Training both with optimal configs...\n")

best_result, best_ppo, best_dqn = (
    find_best_combined(
        env,
        n_train=1000,
        n_eval=100
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ppo_smooth = pd.Series(
    best_ppo.episode_rewards
).rolling(window=50).mean()
dqn_smooth = pd.Series(
    best_dqn.episode_rewards
).rolling(window=50).mean()

ax.plot(
    ppo_smooth, color='gold',
    linewidth=2.5, label='Best PPO'
)
ax.plot(
    dqn_smooth, color='coral',
    linewidth=2, label='Best DQN'
)
ax.set_title(
    'Best PPO vs Best DQN Training Curves',
    fontweight='bold', fontsize=13
)
ax.set_xlabel('Episode')
ax.set_ylabel('Revenue ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    '../results/best_training_curves.png',
    bbox_inches='tight', dpi=150
)
plt.show()
print("✅ Training curves saved!")

In [ ]:
winner = best_result['winner']
ppo_rev = best_result['ppo_revenue']
dqn_rev = best_result['dqn_revenue']
bl_rev  = best_result['baseline_revenue']
ppo_imp = best_result['ppo_vs_bl_pct']
ppo_dqn = best_result['ppo_vs_dqn_pct']

print("╔══════════════════════════════════════════╗")
print("║   WEEK 3 DAY 3 — TUNING COMPLETE! 🔧    ║")
print("╠══════════════════════════════════════════╣")
print("║  PPO BEST CONFIG:                        ║")
print(f"║  LR         : "
      f"{best_config['learning_rate']}"
      f"{'':<26} ║")
print(f"║  Clip Range : "
      f"{best_config['clip_range']}"
      f"{'':<27} ║")
print(f"║  N Epochs   : "
      f"{best_config['n_epochs']}"
      f"{'':<28} ║")
print("╠══════════════════════════════════════════╣")
print("║  FINAL RESULTS:                          ║")
print(f"║  PPO Revenue : ${ppo_rev:.0f}"
      f"{'':<22} ║")
print(f"║  DQN Revenue : ${dqn_rev:.0f}"
      f"{'':<22} ║")
print(f"║  Baseline    : ${bl_rev:.0f}"
      f"{'':<22} ║")
print(f"║  PPO vs DQN  : {ppo_dqn:+.1f}%"
      f"{'':<24} ║")
print(f"║  PPO vs Base : {ppo_imp:+.1f}%"
      f"{'':<24} ║")
print("╠══════════════════════════════════════════╣")
print(f"║  🏆 WINNER: {winner:<30} ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Week 3 Analysis 📊           ║")
print("╚══════════════════════════════════════════╝")